In [1]:
import torch
import torchcrop
from torchcrop.utils.io import make_constant_weather

weather = make_constant_weather(batch_size=2, n_days=150)
model = torchcrop.Lintul5Model()
output = model(weather, start_doy=60)

print(output.yield_)  # [B] final storage-organ biomass (g m-2)
print(output.lai.shape)  # [B, T+1] LAI trajectory
print(output.dvs.shape)  # [B, T+1] development stage trajectory

tensor([244.0944, 244.0944])
torch.Size([2, 151])
torch.Size([2, 151])


In [2]:
import torch
import torch.nn as nn
from torchcrop import Lintul5Model, CropParameters

# Wrap every parameter we want to learn as nn.Parameter. The model reads
# RUE from `ruetb(DVS)` (per-DVS table), so optimising the scalar
# `crop.rue` alone has no effect — make `ruetb` learnable instead (or
# `scale_factor_rue` if you want a single scalar multiplier).
crop = CropParameters().to(dtype=torch.float64)
crop.tsum1 = nn.Parameter(crop.tsum1.detach().clone())
crop.ruetb = nn.Parameter(crop.ruetb.detach().clone())

model = Lintul5Model(crop_params=crop).double()

# Every parameter we want to update must be in the optimizer list — that
# is also what `optimizer.zero_grad()` zeros each iteration.
optimizer = torch.optim.Adam([crop.tsum1, crop.ruetb], lr=1e-1)

# Per-batch target (out.yield_ has shape [B]); avoids a silent broadcast.
observed_yield = torch.tensor([1200.0, 1200.0], dtype=torch.float64)

for i in range(200):
    optimizer.zero_grad()
    out = model(weather.to(torch.float64), start_doy=60)
    loss = ((out.yield_ - observed_yield) ** 2).mean()
    loss.backward()
    optimizer.step()

    if i % 5 == 0:
        print(
            f"Iter {i:2d}: loss={loss.item():9.2f}  "
            f"yield={out.yield_.mean().item():7.2f}  "
            f"tsum1={crop.tsum1.item():6.2f}  "
            f"|∇tsum1|={crop.tsum1.grad.norm().item():.2e}  "
            f"|∇ruetb|={crop.ruetb.grad.norm().item():.2e}"
        )

Iter  0: loss=919990.85  yield= 240.84  tsum1=899.90  |∇tsum1|=4.74e+02  |∇ruetb|=3.02e+05
Iter  5: loss=717020.29  yield= 353.23  tsum1=899.40  |∇tsum1|=7.37e+02  |∇ruetb|=1.43e+05
Iter 10: loss=609267.85  yield= 419.44  tsum1=898.88  |∇tsum1|=8.92e+02  |∇ruetb|=2.31e+05
Iter 15: loss=483805.85  yield= 504.44  tsum1=898.36  |∇tsum1|=8.57e+02  |∇ruetb|=2.65e+05
Iter 20: loss=428719.75  yield= 545.23  tsum1=897.81  |∇tsum1|=1.17e+03  |∇ruetb|=1.79e+05
Iter 25: loss=347473.69  yield= 610.53  tsum1=897.26  |∇tsum1|=1.04e+03  |∇ruetb|=2.56e+05
Iter 30: loss=282320.58  yield= 668.66  tsum1=896.72  |∇tsum1|=7.23e+02  |∇ruetb|=2.35e+05
Iter 35: loss=254077.64  yield= 695.94  tsum1=896.23  |∇tsum1|=7.61e+02  |∇ruetb|=2.02e+05
Iter 40: loss=217708.81  yield= 733.41  tsum1=895.78  |∇tsum1|=7.63e+02  |∇ruetb|=1.61e+05
Iter 45: loss=191454.26  yield= 762.45  tsum1=895.33  |∇tsum1|=7.42e+02  |∇ruetb|=7.81e+04
Iter 50: loss=175693.59  yield= 780.84  tsum1=894.89  |∇tsum1|=8.37e+02  |∇ruetb|=7.13e+04

In [4]:
import torch
import torch.nn as nn
from torchcrop import Lintul5Model, CropParameters

crop = CropParameters().to(dtype=torch.float64)

# Keep original table
ruetb_init = crop.ruetb.detach().clone()

# Fixed DVS column
crop.ruetb_dvs = ruetb_init[:, 0].clone()

# Trainable RUE column
crop.ruetb_rue = nn.Parameter(
    ruetb_init[:, 1].clone()
)

# Rebuild table helper
def build_ruetb():
    return torch.stack(
        [crop.ruetb_dvs, crop.ruetb_rue],
        dim=1
    )

# Inject dynamically before forward
model = Lintul5Model(crop_params=crop).double()

optimizer = torch.optim.Adam(
    [crop.ruetb_rue],
    lr=1e-2
)

observed_yield = torch.tensor(
    [1200.0, 1200.0],
    dtype=torch.float64
)

for i in range(200):

    optimizer.zero_grad()

    # rebuild ruetb with fixed DVS + learned RUE
    crop.ruetb = build_ruetb()

    out = model(weather.to(torch.float64), start_doy=60)

    loss = ((out.yield_ - observed_yield) ** 2).mean()

    loss.backward()

    optimizer.step()

    if i % 5 == 0:
        print(
            f"Iter {i:3d} "
            f"loss={loss.item():10.3f} "
            f"yield={out.yield_.mean().item():8.2f} "
            f"|∇RUE|={crop.ruetb_rue.grad.norm().item():.3e}"
        )

Iter   0 loss=919990.852 yield=  240.84 |∇RUE|=1.172e+05
Iter   5 loss=909816.281 yield=  246.16 |∇RUE|=1.192e+05
Iter  10 loss=898359.977 yield=  252.18 |∇RUE|=1.211e+05
Iter  15 loss=887731.686 yield=  257.80 |∇RUE|=1.233e+05
Iter  20 loss=875727.444 yield=  264.20 |∇RUE|=1.253e+05


KeyboardInterrupt: 

In [5]:
crop.ruetb

tensor([[0.0000, 2.7771],
        [1.0000, 2.7781],
        [1.3000, 3.2205],
        [2.0000, 0.6207]], dtype=torch.float64, grad_fn=<StackBackward0>)